# Week 15 Capstone In-Class Exercise

**CE 310 — Probability & Statistics for Civil & Architectural Engineering**  
University of Arizona — Fall 2026

---

## Your Team's Project

Your team chose its own real-world civil/architectural engineering dataset — any topic, any public source. This notebook applies the full CE 310 toolkit (regression, hypothesis testing, Monte Carlo simulation, bootstrap confidence intervals) to **your** data.

Before writing any code, your team should already have:
1. A dataset (CSV) with enough rows to support these methods (100+ rows recommended)
2. A **numeric predictor variable** (X) and a **numeric outcome variable** (Y) you expect to be related
3. A **grouping variable** that splits your data into two meaningful groups to compare (e.g., weekday/weekend, wet season/dry season, occupied/unoccupied — whatever fits your data)
4. A client scenario and 2–3 questions your team wrote for your client brief (your own brief, not an instructor-provided one)

If you're still choosing a dataset, see the dataset selection guide for what your data needs to support these methods, plus suggested public sources.

---

## Google Colab Users

Upload your team's CSV to Colab session storage before running any cells:
1. Click the folder icon in the left sidebar.
2. Click the upload button and select your CSV file.
3. In Cell 3, fill in `DATASET_NAME`, `DATASET_SOURCE`, `CSV_FILENAME`, and the three column-role variables (`PREDICTOR_COL`, `OUTCOME_COL`, `GROUP_COL`) to match your data.

---

## Grading — This Notebook (30 points, one per team)

This notebook is a **team deliverable** — your team analyses your chosen dataset together and submits one notebook, scored once for all four of you. It is one of three separate capstone deliverables (see note below). **Grading is fully manual** — there's no fixed "correct answer" since every team's dataset is different; a preceptor reads your notebook and evaluates whether you applied each method correctly and interpreted your own results soundly.

| Component | Technical pts | Written pts | Total |
|---|---|---|---|
| A — EDA (ANSWER_A1, A2, A3 + plot + written_A) | 5 | 2 | 7 |
| B — Regression (B1, B2, B3 + bootstrap CI B4, B5 + residuals + written_B) | 7 | 4 | 11 |
| C — Hypothesis Test (ANSWER_C1 + written_C) | 2 | 3 | 5 |
| D — Uncertainty Analysis / Monte Carlo (ANSWER_D1) | 1 | 0 | 1 |
| E — Executive Summary Memo (written_memo) | 0 | 6 | 6 |
| **Notebook total** | **15** | **15** | **30** |

**Your other two capstone deliverables are graded separately, not part of this notebook's score:**
- **Team Capstone Report** (`Capstone/Cap.04`/`Capstone/Cap.05`) — 20 pts, one submission per team, due Sun Dec 6, 11:59 pm
- **Team Presentation** (Week 16 rubric) — 50 pts, scored live during your presentation slot

---
**Before you begin:** Upload your team's CSV to Colab.
1. In Colab: click the **Files** icon (left sidebar) → **Upload to session storage** → select your CSV
2. Confirm the filename shown in the file list matches what you'll set as `CSV_FILENAME` in Cell 3
3. Fill in Cell 3's dataset name/source and column-role variables to match your data
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

%matplotlib inline

In [ ]:
# ── Fill these in to match YOUR team's dataset ──
DATASET_NAME = "Enter a short name for your dataset"       # e.g. "I-10 Tucson Traffic Volume 2023"
DATASET_SOURCE = "Enter where you got it (URL or platform)"  # e.g. "ADOT Open Data Portal"
CSV_FILENAME = "your_dataset.csv"                            # must match the file you uploaded to Colab

PREDICTOR_COL = "Your_Predictor_Column"   # numeric column that predicts your outcome (X)
OUTCOME_COL   = "Your_Outcome_Column"     # numeric column you're trying to explain (Y)
GROUP_COL     = "Your_Grouping_Column"    # categorical/binary column that splits your data into two groups to compare
GROUP_VALUE_A = "GroupA_label"            # the value in GROUP_COL for "Group A" (e.g. "Weekday", "Occupied", "Wet Season")
GROUP_VALUE_B = "GroupB_label"            # the value in GROUP_COL for "Group B" (e.g. "Weekend", "Unoccupied", "Dry Season")

print(f"DATASET_NAME = {DATASET_NAME}")
print(f"DATASET_SOURCE = {DATASET_SOURCE}")

df = pd.read_csv(CSV_FILENAME)

# Standardize your chosen columns into generic X/Y/GROUP so the rest of this
# notebook works regardless of your original column names.
df['X'] = df[PREDICTOR_COL]
df['Y'] = df[OUTCOME_COL]
df['GROUP'] = (df[GROUP_COL] == GROUP_VALUE_A).astype(int)  # 1 = Group A, 0 = Group B

print(df.head())
print(df.info())

In [ ]:
# ── Identify your team ───────────────────────────────────────
# One notebook per team, submitted once. List all four members and run
# this cell — a member who is not listed here cannot be matched to a
# student record and cannot be given the team's grade.
TEAM = [
    ("", ""),   # ("Jordan Reyes", "abc123")
    ("", ""),
    ("", ""),
    ("", ""),
]

for _name, _netid in TEAM:
    print(f"MEMBER = {_name} | {_netid}")

## Section A — Exploratory Data Analysis

In [ ]:
print(df.describe())
print(df[GROUP_COL].value_counts())

ANSWER_A1 = len(df)
print(f"ANSWER_A1 = {ANSWER_A1}")

ANSWER_A2 = round(df['X'].mean(), 2)
print(f"ANSWER_A2 = {ANSWER_A2}")

ANSWER_A3 = round(df['Y'].mean(), 2)
print(f"ANSWER_A3 = {ANSWER_A3}")

In [ ]:
group_colors = {1: 'tomato', 0: 'steelblue'}
point_colors = df['GROUP'].map(group_colors)

fig, ax = plt.subplots(2, 2, figsize=(12, 9))

# YOUR CODE HERE — ax[0,0]: histogram of df['X'] (your predictor)

# YOUR CODE HERE — ax[0,1]: histogram of df['Y'] (your outcome)

# YOUR CODE HERE — ax[1,0]: scatter df['X'] vs df['Y'], colored by group
# Hint: ax[1,0].scatter(df['X'], df['Y'], c=point_colors, alpha=0.3, s=5)

# YOUR CODE HERE — ax[1,1]: box plot of df['Y'] by GROUP_COL
# Hint: df.boxplot(column='Y', by=GROUP_COL, ax=ax[1,1])

plt.suptitle(f"{DATASET_NAME} — Data Profile", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation (written_A):** What patterns do you see in your data? Mention how X and Y appear to relate, any grouping differences, and any surprises. (2–3 sentences)

YOUR OBSERVATION HERE

## Section B — Regression Analysis

In [ ]:
model1 = smf.ols('Y ~ X', data=df).fit()
print(model1.summary())

In [ ]:
model2 = smf.ols('Y ~ X + GROUP', data=df).fit()
print(model2.summary())

ANSWER_B1 = round(model2.params['X'], 4)
print(f"ANSWER_B1 = {ANSWER_B1}")

ANSWER_B2 = round(model2.params['GROUP'], 4)
print(f"ANSWER_B2 = {ANSWER_B2}")

ANSWER_B3 = round(model2.rsquared, 4)
print(f"ANSWER_B3 = {ANSWER_B3}")

### Section B4–B5 — Bootstrap Confidence Interval for Your Predictor's Slope

In Week 14 you bootstrapped a regression slope. Apply the same method to your own model.

**Why bootstrap instead of the analytical OLS CI?** The OLS interval assumes the residuals are independent and identically distributed. Real engineering data often breaks that assumption — repeated measurements at the same site, observations ordered in time, or readings clustered by season, operator or instrument are all correlated in ways OLS cannot see, and its interval comes out too narrow as a result. Resampling rows makes no such assumption, so where the two disagree, the bootstrap is the more honest of the two.

Both intervals are printed below. If yours differ noticeably, say in your Section E memo what about *your* data would explain it — and if they agree closely, say that too, because it is evidence your rows really are independent.

Use `np.random.default_rng(42)` with `N_BOOT = 1000`. Resample **rows** (not residuals) with replacement, refit Model 2, and collect the coefficient on `X` each time.

In [ ]:
N_BOOT = 1000
rng_boot = np.random.default_rng(42)
n = len(df)
boot_slopes = []
for _ in range(N_BOOT):
    idx = rng_boot.integers(0, n, n)
    sample = df.iloc[idx]
    m = smf.ols('Y ~ X + GROUP', data=sample).fit()
    boot_slopes.append(m.params['X'])

boot_slopes = np.array(boot_slopes)
ANSWER_B4_boot_lower = round(float(np.percentile(boot_slopes, 2.5)), 4)
ANSWER_B5_boot_upper = round(float(np.percentile(boot_slopes, 97.5)), 4)
print(f"Bootstrap 95% CI for X slope: [{ANSWER_B4_boot_lower:.4f}, {ANSWER_B5_boot_upper:.4f}]")
print(f"Analytical OLS CI:            [{round(float(model2.conf_int().loc['X', 0]), 4):.4f}, "
      f"{round(float(model2.conf_int().loc['X', 1]), 4):.4f}]")
print(f"ANSWER_B4_boot_lower = {ANSWER_B4_boot_lower}")
print(f"ANSWER_B5_boot_upper = {ANSWER_B5_boot_upper}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_slopes, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(ANSWER_B4_boot_lower, color='orange', linestyle='--', linewidth=1.5,
           label=f'2.5th percentile = {ANSWER_B4_boot_lower:.4f}')
ax.axvline(ANSWER_B5_boot_upper, color='orange', linestyle='--', linewidth=1.5,
           label=f'97.5th percentile = {ANSWER_B5_boot_upper:.4f}')
ax.axvline(ANSWER_B1, color='red', linewidth=1.5, label=f'Observed slope = {ANSWER_B1}')
ax.set_xlabel('Bootstrap Slope (Y per unit X)')
ax.set_ylabel('Frequency')
ax.set_title(f'{DATASET_NAME} — Bootstrap Distribution of X Slope (N_BOOT={N_BOOT})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# YOUR CODE HERE — left plot: fitted values vs residuals
# Hint: axes[0].scatter(model2.fittedvalues, model2.resid, alpha=0.3, s=5)
# Add a horizontal line at y=0: axes[0].axhline(0, color='red', linewidth=1)
# Label axes appropriately

# YOUR CODE HERE — right plot: Q-Q plot of residuals
# Hint: stats.probplot(model2.resid, plot=axes[1])

plt.suptitle(f"{DATASET_NAME} — Model 2 Residual Diagnostics", fontsize=13)
plt.tight_layout()
plt.show()

**Interpretation (written_B):** Compare Model 1 and Model 2. Does adding GROUP improve the fit? Is the bootstrap CI (B4–B5) wider or narrower than the analytical OLS CI, and why might that be true for your data? Does the sign of ANSWER_B2 make physical/practical sense for your two groups? If it's counterintuitive, explain why. (3–4 sentences)

YOUR INTERPRETATION HERE

## Section C — Hypothesis Test: Difference Between Your Two Groups

In [ ]:
group_a_vals = df.loc[df['GROUP'] == 1, 'Y']
group_b_vals = df.loc[df['GROUP'] == 0, 'Y']

t_stat, p_val = stats.ttest_ind(group_a_vals, group_b_vals)

print(f"t-statistic = {t_stat:.4f}")
print(f"p-value     = {p_val:.6f}")
print(f"Mean Y ({GROUP_VALUE_A}) = {group_a_vals.mean():.3f}")
print(f"Mean Y ({GROUP_VALUE_B}) = {group_b_vals.mean():.3f}")

ANSWER_C1 = round(group_a_vals.mean(), 3)
print(f"ANSWER_C1 = {ANSWER_C1}")

**Critical thinking:** The t-test does not control for your predictor variable (X). If Group A and Group B happen to coincide with systematically different X values, that confounding could explain part of the gap between the raw mean difference here and the regression coefficient ANSWER_B2. Does this apply to your data? Discuss briefly.

YOUR T-TEST INTERPRETATION HERE

## Section D — Uncertainty Analysis (Monte Carlo Simulation)

Use Monte Carlo simulation to propagate uncertainty in your predictor (X) into uncertainty in your outcome (Y), using your Model 2 regression as the underlying relationship — same method as Week 13, applied to your own variables.

Set `X_UNCERTAINTY` to a value that makes sense for your predictor (e.g., how much it might reasonably vary from one period to the next — check `df['X'].std()` for a reference scale before choosing). You'll justify this choice in your Section E memo.

In [ ]:
ANSWER_D1 = round(df['Y'].mean(), 2)
print(f"ANSWER_D1 = {ANSWER_D1}")
print(f"Deterministic baseline (mean Y): {ANSWER_D1}")

N = 10_000
rng = np.random.default_rng(42)

X_UNCERTAINTY = 2.0  # <-- set this to a value that makes sense for YOUR predictor variable (see markdown above)

delta_X = rng.normal(0, X_UNCERTAINTY, N)
Y_samples = (model2.params['Intercept']
             + (df['X'].mean() + delta_X) * model2.params['X']
             + df['GROUP'].mean() * model2.params['GROUP'])

print(f"Monte Carlo Y — Mean : {Y_samples.mean():,.2f}")
print(f"Monte Carlo Y — Std  : {Y_samples.std():,.2f}")
print(f"Monte Carlo Y — P10  : {np.percentile(Y_samples, 10):,.2f}")
print(f"Monte Carlo Y — P90  : {np.percentile(Y_samples, 90):,.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(Y_samples, bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(Y_samples.mean(), color='red', linewidth=1.5, label=f"Mean = {Y_samples.mean():,.2f}")
ax.axvline(np.percentile(Y_samples, 10), color='orange', linestyle='--', linewidth=1.2, label='P10')
ax.axvline(np.percentile(Y_samples, 90), color='orange', linestyle='--', linewidth=1.2, label='P90')
ax.set_xlabel(f"{OUTCOME_COL}")
ax.set_ylabel("Frequency")
ax.set_title(f"{DATASET_NAME} — Monte Carlo Simulated Outcome Distribution (N={N:,})")
ax.legend()
plt.tight_layout()
plt.show()

## Section E — Executive Summary

Write a **3-paragraph executive summary memo** addressed to your dataset's client (the stakeholder from your `15.5_client-briefs.pdf` brief). Replace the placeholder cell below.

- **Paragraph 1 — Data Profile:** Describe your dataset (ANSWER_A1 rows), the mean of your predictor (ANSWER_A2) and outcome (ANSWER_A3), and the X-slope (ANSWER_B1) in plain language your client would understand.
- **Paragraph 2 — Group Finding:** Explain what the GROUP coefficient (ANSWER_B2) means operationally for your two groups. If the coefficient is counterintuitive, explain the practical reason.
- **Paragraph 3 — Uncertainty & Recommendation:** Summarize the Monte Carlo P10–P90 range, briefly justify the `X_UNCERTAINTY` value you chose, and propose one actionable measure supported by your analysis.

YOUR EXECUTIVE SUMMARY HERE

---
## Before You Submit

- [ ] All four members listed in the team cell, with NetIDs, and that cell run
- [ ] Every cell run in order, top to bottom (Runtime → Run all) — no errors, no cell left unrun
- [ ] Every `ANSWER_` line prints a value, and no `print(f"ANSWER_... = ...")` line edited or deleted
- [ ] Cell 3's dataset name, source, and column-role variables are filled in for your actual data
- [ ] Executive Summary is filled in (not placeholder text)
- [ ] Downloaded as `.ipynb` — not PDF, not `.py`
- [ ] Renamed `lastname1_lastname2_lastname3_lastname4_capstone_notebook.ipynb` (alphabetical by last name) and uploaded once, by one member, to the `Week15_Capstone_Notebook` dropbox